#**Домашнее задание №2**
##**Выполнила:** Шумилова Мария Константиновна 409920 U3310
##**Проверила:** Желтова Кристина Анатольевна

## **Загрузка данных**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA

%matplotlib inline
sns.set_style("whitegrid")

# export KAGGLE_API_TOKEN=KGAT_e32f1e27c3076e0083f2ef1a7904bc58
%pip install opendatasets -q
import opendatasets as od

od.download("https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/data")

csv_files = glob.glob("ibm-hr-analytics-attrition-dataset/*.csv")

df = pd.read_csv(csv_files[0])

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: bulk2403
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset


100%|██████████| 50.1k/50.1k [00:00<00:00, 24.7MB/s]

### **Зафиксируем `random_state`**

In [7]:
import random
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

Для воспроизводимости решения был зафиксирован параметр `random_state`, а также установлены одинаковые значения генератора случайных чисел в Python и NumPy. Это позволяет получать одинаковые результаты при повторном запуске ноутбука.

## **1. Разбиение на обучающую и тестовую выборки**

In [8]:
X = df.drop(columns=["Attrition"])
y = df["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Размер обучающей выборки:", X_train.shape)
print("Размер тестовой выборки:", X_test.shape)

print("\nРаспределение классов в train:")
print(y_train.value_counts(normalize=True).round(3))

print("\nРаспределение классов в test:")
print(y_test.value_counts(normalize=True).round(3))

Размер обучающей выборки: (1176, 34)
Размер тестовой выборки: (294, 34)

Распределение классов в train:
Attrition
No     0.838
Yes    0.162
Name: proportion, dtype: float64

Распределение классов в test:
Attrition
No     0.84
Yes    0.16
Name: proportion, dtype: float64


Для дальнейшего обучения модели данные были разделены на обучающую и тестовую выборки в соотношении 80/20.  
Так как целевая переменная является бинарной и классы в датасете несбалансированы, при разбиении была использована стратификация по целевой переменной. Это позволяет сохранить исходное соотношение классов в обеих выборках и получить более корректную оценку качества модели.

## **2. Качество константного предсказания**

In [9]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

dummy_model = DummyClassifier(strategy="most_frequent", random_state=42)
dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

dummy_accuracy = accuracy_score(y_test, y_pred_dummy)
dummy_balanced_accuracy = balanced_accuracy_score(y_test, y_pred_dummy)
dummy_precision = precision_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)
dummy_recall = recall_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)
dummy_f1 = f1_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)

print("Качество константного бейзлайна:")
print(f"Accuracy: {dummy_accuracy:.3f}")
print(f"Balanced Accuracy: {dummy_balanced_accuracy:.3f}")
print(f"Precision: {dummy_precision:.3f}")
print(f"Recall: {dummy_recall:.3f}")
print(f"F1-score: {dummy_f1:.3f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred_dummy))

Качество константного бейзлайна:
Accuracy: 0.840
Balanced Accuracy: 0.500
Precision: 0.000
Recall: 0.000
F1-score: 0.000

Classification report:
              precision    recall  f1-score   support

          No       0.84      1.00      0.91       247
         Yes       0.00      0.00      0.00        47

    accuracy                           0.84       294
   macro avg       0.42      0.50      0.46       294
weighted avg       0.71      0.84      0.77       294


Confusion matrix:
[[247   0]
 [ 47   0]]


**Какое решение использовали**

В качестве базового решения был использован `DummyClassifier` со стратегией `most_frequent`, который всегда предсказывает наиболее частотный класс. В данном наборе данных таким классом является `No`, то есть сотрудник не уволится. Использование такой модели позволяет получить минимальный ориентир качества, с которым затем можно сравнивать более сложные алгоритмы.

---

**Какие метрики использовали**

Для оценки качества были выбраны метрики `accuracy`, `balanced accuracy`, `precision`, `recall` и `F1-score`. Метрика `accuracy` показывает общую долю правильных ответов, однако в условиях дисбаланса классов может давать завышенную оценку качества. Поэтому дополнительно используется `balanced accuracy`, учитывающая качество предсказаний по каждому классу. Метрики `precision`, `recall` и `F1-score` рассчитываются для класса `Yes`, так как именно он является целевым и наиболее важным в задаче.

---

**Результаты и вывод**

Полученные значения показывают, что константный классификатор действительно предсказывает только наиболее частотный класс, которым в данном случае является `No`. Именно поэтому значение `accuracy` получилось достаточно высоким — 0.84. Однако такая точность здесь не отражает реального качества модели, потому что класс `No` в датасете существенно преобладает. Более показательным результатом является `balanced accuracy = 0.50`, что соответствует фактически случайному угадыванию. Это означает, что модель хорошо справляется только с одним классом и полностью игнорирует второй.

Это также подтверждается `classification report` и матрицей ошибок. Для класса `No` модель получила высокий `recall = 1.00`, то есть все объекты этого класса были правильно определены. В то же время для класса `Yes` все метрики равны нулю: модель не смогла обнаружить ни одного сотрудника, который действительно уволился. Матрица ошибок показывает, что все 47 объектов класса `Yes` были ошибочно отнесены к `No`.

**Таким образом, константный бейзлайн не решает поставленную задачу, а лишь служит минимальной точкой отсчёта для сравнения с более содержательными моделями.**

## **3. Измерение качества на отложенной выборке**

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

dummy_model = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred_dummy)
balanced_acc = balanced_accuracy_score(y_test, y_pred_dummy)
precision = precision_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)
recall = recall_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)
f1 = f1_score(y_test, y_pred_dummy, pos_label="Yes", zero_division=0)

print(f"Accuracy: {accuracy:.3f}")
print(f"Balanced Accuracy: {balanced_acc:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1-score: {f1:.3f}")

Accuracy: 0.840
Balanced Accuracy: 0.500
Precision: 0.000
Recall: 0.000
F1-score: 0.000


Качество модели было измерено с использованием ранее выбранных метрик: `accuracy`, `balanced accuracy`, `precision`, `recall` и `F1-score`. Результаты на тестовой выборке совпадают с ранее полученными значениями для константного бейзлайна, так как модель не обучается на признаках и всегда предсказывает один и тот же класс.

Полученные значения подтверждают, что модель не способна выявлять сотрудников, склонных к увольнению: метрики для класса `Yes` равны нулю. Это ещё раз показывает, что данный бейзлайн может использоваться только как отправная точка для дальнейшего улучшения модели.и может использоваться только как базовая точка отсчёта.

## **4. Общий вывод**

В рамках второго задания были выполнены основные шаги подготовки к построению модели машинного обучения. Сначала было произведено разбиение данных на обучающую и тестовую выборки в соотношении 80/20. Для сохранения исходного распределения классов была использована стратификация, что особенно важно в условиях дисбаланса целевой переменной.

Далее был построен и оценён константный бейзлайн с помощью `DummyClassifier`, который всегда предсказывает наиболее частотный класс. Такой подход позволяет получить минимальный ориентир качества, с которым можно сравнивать более сложные модели. Для оценки были использованы метрики `accuracy`, `balanced accuracy`, `precision`, `recall` и `F1-score`, поскольку они позволяют учитывать особенности несбалансированной классификационной задачи.

Результаты показали, что высокая accuracy у базовой модели не отражает реального качества, так как она не способна выявлять редкий целевой класс `Yes`. Более корректная метрика `balanced accuracy` оказалась равна 0.50, а `precision`, `recall` и `F1-score` для класса увольнения — нулевыми. Это подтвердило, что простое предсказание самого частотного класса не подходит для решения данной задачи.

Таким образом, в ходе работы был получен корректный базовый уровень качества и подготовлена основа для дальнейшего сравнения с более содержательными моделями машинного обучения.